# Assignment: Wikipedia Page Views Pipeline

Build a data pipeline for Wikipedia page views, similar to what we did for edits.

## Overview

You will:
1. Create a notebook that extracts top viewed Wikipedia pages
2. Create Athena tables to query the data
3. Deploy as a Lambda function
4. Schedule with EventBridge


## Part 1: Create the Notebook

Create a new notebook in `/homework/extract_views.ipynb` that extracts, transforms, and loads page view data.


In [ ]:
import datetime
import json
import boto3
import requests

In [ ]:
USERNAME = "catheloni"

date1 = datetime.datetime.strptime("2024-11-04", "%Y-%m-%d").strftime("%Y/%m/%d")
date2 = datetime.datetime.strptime("2024-11-06", "%Y-%m-%d").strftime("%Y/%m/%d")
date3 = datetime.datetime.strptime("2025-01-21", "%Y-%m-%d").strftime("%Y/%m/%d")

print(date1)

2024/11/04


In [5]:
date1 = datetime.datetime.strptime("2024-11-04", "%Y-%m-%d").strftime("%Y/%m/%d")
current_time = current_time = datetime.datetime.now(datetime.timezone.utc)

url = f"https://wikimedia.org/api/rest_v1/metrics/pageviews/top/en.wikipedia.org/all-access/{date1}"

print(f"Requesting REST API URL: {url}")

# Make the API request
wiki_server_response = requests.get(url, headers={"User-Agent": "curl/7.68.0"}) # wikipedia doesn't have an anti-crawler policy but you need to specify who you are
wiki_response_status = wiki_server_response.status_code # must be 200 for http OK
wiki_response_body = wiki_server_response.text

print(f"Wikipedia REST API Response body: {wiki_response_body[:500]}...")
print(f"Wikipedia REST API Response Code: {wiki_response_status}")

# Validate response
if wiki_response_status != 200:
    raise Exception(f"Received non-OK status code from Wiki Server: {wiki_response_status}") # will stop execution
print(f"Successfully retrieved Wikipedia data, content-length: {len(wiki_response_body)}")

wiki_response_parsed = wiki_server_response.json()

json_lines = ""

for i in wiki_response_parsed["items"][0]["articles"][:10]:
    record = {
        "title": i["article"],
        "views": i["views"],
        "rank": i["rank"],
        "date": date1,
        "retrieved_at": current_time.replace(tzinfo=None).isoformat(),
        }
    json_lines += json.dumps(record) + "\n"


Requesting REST API URL: https://wikimedia.org/api/rest_v1/metrics/pageviews/top/en.wikipedia.org/all-access/2024/11/04
Wikipedia REST API Response body: {"items":[{"project":"en.wikipedia","access":"all-access","year":"2024","month":"11","day":"04","articles":[{"article":"Main_Page","views":4745899,"rank":1},{"article":"Special:Search","views":2587172,"rank":2},{"article":"Quincy_Jones","views":1063149,"rank":3},{"article":"Wikipedia:Featured_pictures","views":709954,"rank":4},{"article":"2024_United_States_presidential_election","views":584016,"rank":5},{"article":"Kamala_Harris","views":278609,"rank":6},{"article":"Rashida_Jones","views":26843...
Wikipedia REST API Response Code: 200
Successfully retrieved Wikipedia data, content-length: 57399


In [6]:
# /pageviews/top/en.wikipedia.org/all-access/date.strftime("%Y/%m/%d")

wiki_server_response = []

urls = [f"https://wikimedia.org/api/rest_v1/metrics/pageviews/top/en.wikipedia.org/all-access/{date1}",
       f"https://wikimedia.org/api/rest_v1/metrics/pageviews/top/en.wikipedia.org/all-access/{date2}",
       f"https://wikimedia.org/api/rest_v1/metrics/pageviews/top/en.wikipedia.org/all-access/{date3}"]


for u in urls:
    response = requests.get(u, headers={"User-Agent": "curl/7.68.0"})
    wiki_server_response.append(response)

In [7]:
wiki_response_parsed = []

for i,j in enumerate(wiki_server_response):
    wiki_response_parsed.append(wiki_server_response[i].json())

In [8]:
#wiki_response_parsed["items"][0]["articles"]

wiki_response_parsed[0]["items"][0]["articles"]
fdate = f"{wiki_response_parsed[0]["items"][0]["year"]}/{wiki_response_parsed[0]["items"][0]["month"]}/{wiki_response_parsed[0]["items"][0]["day"]}"
print(fdate)
date = datetime.datetime.strptime(fdate, "%Y/%m/%d").strftime("%Y/%m/%d")

2024/11/04


In [9]:
json_lines = ""

for response in wiki_response_parsed:
    item = response["items"][0]

    fdate = f"{item['year']}/{item['month']}/{item['day']}"
    date = datetime.datetime.strptime(fdate, "%Y/%m/%d").strftime("%Y/%m/%d")

    # Top 10 articles
    for art in item["articles"][:10]:
        record = {
            "title": art["article"],
            "views": art["views"],
            "rank": art["rank"],
            "date": date,
            "retrieved_at": current_time.replace(tzinfo=None).isoformat(),
        }

        json_lines += json.dumps(record) + "\n"


In [ ]:

## Upload to S3

# create new bucket
S3_WIKI_BUCKET = "catheloni-wikidata"
s3 = boto3.client("s3")

bucket_names = [bucket["Name"] for bucket in s3.list_buckets()["Buckets"]]
if S3_WIKI_BUCKET not in bucket_names:
    s3.create_bucket(
        ACL= "private",
        Bucket = "raw-views",
        CreateBucketConfiguration={
            "LocationConstraint": "eu-west-1"
        } )

    print(f"Created new bucket: {S3_WIKI_BUCKET}")
else:
    print(f"Using existing bucket: {S3_WIKI_BUCKET}")


Using existing bucket: catheloni-wikidata


In [126]:
# code from class

#print(date1.datetime.strptime('%Y-%m-%d'))
date01 = datetime.datetime.strptime("2024-11-04", "%Y-%m-%d").strftime("%Y-%m-%d")
date02 = datetime.datetime.strptime("2024-11-06", "%Y-%m-%d").strftime("%Y-%m-%d")
date03 = datetime.datetime.strptime("2025-01-21", "%Y-%m-%d").strftime("%Y-%m-%d")

date_list02 = [date01, date02, date03]

In [127]:
s3_key = []

for i in date_list02:
    s3_key.append(f"raw-views/raw-views-{i}.json")

In [130]:
for i in date_list02:
    s3_key = f"raw-views/raw-views-{i}.json"
    s3.put_object(
    Bucket=S3_WIKI_BUCKET,
    Key=s3_key,
    Body=json_lines.encode("utf-8"))
    print(f"Uploaded {len(top_edits)} records to s3://{S3_WIKI_BUCKET}/{s3_key}")


Uploaded 3 records to s3://catheloni-wikidata/raw-views/raw-views-2024-11-04.json
Uploaded 3 records to s3://catheloni-wikidata/raw-views/raw-views-2024-11-06.json
Uploaded 3 records to s3://catheloni-wikidata/raw-views/raw-views-2025-01-21.json
